# Clasificación de sentimiento con BERT
### Enfoque 1 de 3 — BERT como tokenizador

**Universidad de Santiago de Chile (USACH)**  
Diplomado de Inteligencia artificial aplicada a la industria tecnológica  
Autor: Diego Fischer

BERT se usa **únicamente como tokenizador** (su vocabulario WordPiece). El clasificador es una CNN con una capa de *embeddings* entrenada desde cero; BERT no forma parte de la red. Con KerasHub el tokenizador es `keras_hub.models.BertTokenizer`.

> Los tres cuadernos de esta serie resuelven la misma tarea (clasificación binaria de sentimiento sobre el dataset Sentiment140) y se diferencian únicamente en el rol que cumple BERT dentro del modelo.

> **Migrado a KerasHub.** El paquete `bert-for-tf2` quedó obsoleto y no funciona con Keras 3 (el *default* en Colab actual). Aquí usamos [KerasHub](https://keras.io/keras_hub/), la biblioteca nativa de Keras 3 para modelos preentrenados.

---

# Fase 1: Importar las dependencias

Instalamos **KerasHub**. Requiere Keras 3, que ya viene por defecto en Colab, así que no hace falta reiniciar el entorno.

In [2]:
!pip install -q -U keras-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.26.0 requires keras-hub==0.26.0, but you have keras-hub 0.29.1 which is incompatible.


Importamos las librerías. Fijamos TensorFlow como *backend* de Keras 3 **antes** de importar `keras`. `keras_hub` nos da el tokenizador y el modelo BERT preentrenados.

In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # backend de Keras 3

import numpy as np
import math
import re
import pandas as pd
from bs4 import BeautifulSoup
import random

import tensorflow as tf
import keras
import keras_hub
from tensorflow.keras import layers

from google.colab import drive

# Preset de BERT preentrenado (equivalente a bert_en_uncased_L-12_H-768_A-12)
BERT_PRESET = "bert_base_en_uncased"

Montamos Google Drive para leer el dataset.

In [3]:
!wget -q -O data.zip\
https://www.dropbox.com/scl/fi/9wx7mfq1uwil3dnaw2k6x/trainingandtestdata.zip?rlkey=6fitfib2ye55437knbjo8uzwc&st=d5oecd16&dl=0
!unzip -qq data.zip
!rm data.zip

# Fase 2: Preprocesamiento de datos

## Carga de los ficheros

Leemos el CSV de entrenamiento de Sentiment140 (seis columnas; `encoding='latin1'` por los caracteres no UTF-8 de los tweets).

In [4]:
cols = ["sentiment", "id", "date", "query", "user", "text"]
data = pd.read_csv(
    "/content/trainingandtestdata/training.1600000.processed.noemoticon.csv",
    header=None,
    names=cols,
    engine="python",
    encoding="latin1"
)

Conservamos solo las columnas útiles: `sentiment` y `text`.

In [5]:
data.drop(["id", "date", "query", "user"], axis=1, inplace=True)

## Limpieza

Función que limpia cada tweet: quita el HTML, las menciones `@usuario`, las URLs y todo lo que no sean letras o signos básicos, y colapsa los espacios sobrantes.

In [6]:
def clean_tweet(tweet):
    tweet = BeautifulSoup(tweet, "lxml").get_text()
    # Eliminar el @ y su mención
    tweet = re.sub(r"@[A-Za-z0-9]+", ' ', tweet)
    # Eliminar los links de la URL
    tweet = re.sub(r"https?://[A-Za-z0-9./]+", ' ', tweet)
    # Conservar solo las letras y algunos signos
    tweet = re.sub(r"[^a-zA-Z.!?']", ' ', tweet)
    # Eliminar espacios en blanco adicionales
    tweet = re.sub(r" +", ' ', tweet)
    return tweet

Aplicamos la limpieza a todos los tweets.

In [7]:
data_clean = [clean_tweet(tweet) for tweet in data.text]

Extraemos las etiquetas. En Sentiment140 el positivo viene como `4`; lo pasamos a `1` para tener etiquetas binarias **0 (negativo)** / **1 (positivo)**.

In [8]:
data_labels = data.sentiment.values
data_labels[data_labels == 4] = 1

## Tokenización

Cargamos el **tokenizador WordPiece de BERT** con KerasHub. Reemplaza al `FullTokenizer` de `bert-for-tf2`. `vocabulary_size()` nos da el tamaño del vocabulario, que necesitará la capa `Embedding`.

In [9]:
tokenizer = keras_hub.models.BertTokenizer.from_preset(BERT_PRESET)
VOCAB_SIZE = tokenizer.vocabulary_size()
print("Tamaño del vocabulario:", VOCAB_SIZE)

Tamaño del vocabulario: 30522


Codificamos cada oración como una secuencia de enteros. En este enfoque **no** agregamos los tokens `[CLS]`/`[SEP]`, porque solo nos interesan los IDs para alimentar la CNN.

> Nota: el bucle codifica oración por oración y sobre 1,6 M de tweets puede tardar. Si lo necesitas más rápido, `tokenizer(data_clean)` tokeniza el corpus completo en una sola llamada (devuelve un `RaggedTensor`).

In [ ]:
def encode_sentence(sent):
    return tokenizer(sent).numpy().tolist()

data_inputs = [encode_sentence(sentence) for sentence in data_clean]

## Creación del dataset

Creamos *padded batches*: ordenamos las frases por longitud, descartamos las muy cortas y rellenamos cada lote de forma independiente para minimizar el *padding*.

In [ ]:
data_with_len = [[sent, data_labels[i], len(sent)]
                 for i, sent in enumerate(data_inputs)]
random.shuffle(data_with_len)
data_with_len.sort(key=lambda x: x[2])
sorted_all = [(sent_lab[0], sent_lab[1])
              for sent_lab in data_with_len if sent_lab[2] > 7]

In [ ]:
all_dataset = tf.data.Dataset.from_generator(
    lambda: sorted_all, output_types=(tf.int32, tf.int32))

In [ ]:
BATCH_SIZE = 32
all_batched = all_dataset.padded_batch(BATCH_SIZE, padded_shapes=((None,), ()))

NB_BATCHES = math.ceil(len(sorted_all) / BATCH_SIZE)
NB_BATCHES_TEST = NB_BATCHES // 10
all_batched.shuffle(NB_BATCHES)
test_dataset = all_batched.take(NB_BATCHES_TEST)
train_dataset = all_batched.skip(NB_BATCHES_TEST)

# Fase 3: Construcción del modelo

La **CNN para texto (DCNN)**: una capa de *embeddings* propia, tres convoluciones 1D en paralelo (bi/tri/tetragramas), *max pooling*, una densa con *dropout* y la salida sigmoide. BERT no aparece aquí.

In [ ]:
class DCNN(tf.keras.Model):
    def __init__(self, vocab_size, emb_dim=128, nb_filters=50, FFN_units=512,
                 nb_classes=2, dropout_rate=0.1, name="dcnn"):
        super(DCNN, self).__init__(name=name)
        self.embedding = layers.Embedding(vocab_size, emb_dim)
        self.bigram   = layers.Conv1D(nb_filters, 2, padding="valid", activation="relu")
        self.trigram  = layers.Conv1D(nb_filters, 3, padding="valid", activation="relu")
        self.fourgram = layers.Conv1D(nb_filters, 4, padding="valid", activation="relu")
        self.pool = layers.GlobalMaxPool1D()
        self.dense_1 = layers.Dense(units=FFN_units, activation="relu")
        self.dropout = layers.Dropout(rate=dropout_rate)
        if nb_classes == 2:
            self.last_dense = layers.Dense(units=1, activation="sigmoid")
        else:
            self.last_dense = layers.Dense(units=nb_classes, activation="softmax")

    def call(self, inputs, training=False):
        x = self.embedding(inputs)
        x_1 = self.pool(self.bigram(x))
        x_2 = self.pool(self.trigram(x))
        x_3 = self.pool(self.fourgram(x))
        merged = tf.concat([x_1, x_2, x_3], axis=-1)
        merged = self.dense_1(merged)
        merged = self.dropout(merged, training=training)
        return self.last_dense(merged)

# Fase 4: Entrenamiento

Hiperparámetros del modelo.

In [ ]:
EMB_DIM = 200
NB_FILTERS = 100
FFN_UNITS = 256
NB_CLASSES = 2
DROPOUT_RATE = 0.2
NB_EPOCHS = 5

Instanciamos y compilamos el modelo.

In [ ]:
Dcnn = DCNN(vocab_size=VOCAB_SIZE, emb_dim=EMB_DIM, nb_filters=NB_FILTERS,
            FFN_units=FFN_UNITS, nb_classes=NB_CLASSES, dropout_rate=DROPOUT_RATE)

Dcnn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

Guardamos los pesos en Drive al final de cada época con un *callback* de Keras 3.

In [ ]:
checkpoint_path = "ckpt_bert_tok.weights.h5"
ckpt_cb = keras.callbacks.ModelCheckpoint(checkpoint_path, save_weights_only=True)

Entrenamos.

In [ ]:
Dcnn.fit(train_dataset, epochs=NB_EPOCHS, callbacks=[ckpt_cb])

# Fase 5: Evaluación

In [ ]:
results = Dcnn.evaluate(test_dataset)
print(results)

Probamos el modelo con frases nuevas.

In [ ]:
def get_prediction(sentence):
    tokens = encode_sentence(sentence)
    inputs = tf.expand_dims(tokens, 0)
    output = Dcnn(inputs, training=False)
    sentiment = math.floor(float(output) * 2)
    etiqueta = "Positivo" if sentiment == 1 else "Negativo"
    print("Salida del modelo: {}\nSentimiento predicho: {}.".format(float(output), etiqueta))

get_prediction("This movie was pretty interesting.")
get_prediction("I'd rather not do that again.")

---
## Licencia

[![Licencia Creative Commons](https://i.creativecommons.org/l/by/4.0/88x31.png)](https://creativecommons.org/licenses/by/4.0/deed.es)

Esta obra está licenciada bajo una [Licencia Creative Commons Atribución 4.0 Internacional (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/deed.es).

**Atribución:** Diego Fischer — Universidad de Santiago de Chile (USACH).